# 03 · ML benchmark
**What it is:** three machine-learning models trained to recognise the 48 hours before a failure, using every
feature our rules could use. They serve two purposes:
1. **Ideas.** Which inputs does a flexible model lean on? This fed the brainstorm in notebook 02.
2. **A ceiling.** If a model with every feature cannot beat our simple rules on unseen machines, scored the same
   way, the rules are not leaving much on the table, and they are far easier to explain and trust.

**It is not a detector:** the brief asks for simple fixed rules, and none of the kept precursors is an ML model.

The simple rules are tested in notebook 04; notebook 06, section 8 puts the models and the rules side by side.

**Code:** `feature_cube` and `train_and_score` in `src/evaluation.py` §8. This notebook trains the models live
(~2 minutes).

### What this notebook answers from the brief
| Brief asks for | Where |
|---|---|
| "Use whatever helps you generate ideas: domain intuition, an LLM, **an ML model**, or a mix" (Task 2) | section 5 (what the models lean on, and how that shaped the candidates) |
| "Clear reasoning and honest statistics over a high model score" | section 4 (same metrics and same holdout machines as the rules); notebook 06, section 8 |
| Oden's workflow, step 2: "reduce the search space with ML" | this notebook as a whole |

In [1]:
# --- Setup ----------------------------------------------------------------------------
# Libraries, plus the project's helper files in ../src (data.py, detectors.py, evaluation.py).
# Every analysis function this notebook calls lives there, in numbered sections (§1 ... §9).
import sys, time, warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")  # harmless: kagglehub's progress bar

import numpy as np
import pandas as pd
import plotly.io as pio

# Two outputs per chart: Plotly's own MIME type (rendered natively by VS Code and JupyterLab, works
# offline) plus an HTML fallback for classic Jupyter (loads plotly.js from a CDN).
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_colwidth", 120, "display.width", 200)  # show long rule texts in full

ROOT = Path.cwd().parent            # project root (this notebook lives in notebooks/)
sys.path.insert(0, str(ROOT / "src"))  # so `from data import ...` finds ../src/data.py

In [2]:
# --- Code used in this notebook (from ../src) --------------------------------------------
from data import (                            # src/data.py
    Config, load_panel, dev_holdout_split,    #   §1-§2 settings, data, dev/holdout machines
    build_masks,                              #   §3 the scoring window and "normal operation"
)
from detectors import FeatureBank, thresholds # src/detectors.py §4: features (cached); percentile thresholds
from evaluation import (                      # src/evaluation.py
    feature_cube, train_and_score,            #   §8 the ML feature table; train a model and score every hour
    machine_stats, summarize, bootstrap,      #   §6 lift / precision / coverage, cluster-bootstrap intervals
    lead_times,                               #   §7 how early the first alert comes before each failure
    save_results,                             #   §9 save this notebook's key results for notebook 06
)

## 1 · Data, window and machines
Everything matches the rules' setup exactly, so the comparison is fair:
* **the same window:** the 48 h before each failure, failure hour excluded (`Config`: H = 0, W = 48);
* **the same definition of normal operation;**
* **the same 50 dev machines** for training and the same **50 holdout machines** for scoring.

In [3]:
panel = load_panel()
cfg = Config()                                              # H=0, W=48, event="any"
dev, hold = dev_holdout_split(panel, cfg.dev_fraction, cfg.seed)
bank = FeatureBank(panel)
mk = build_masks(panel, cfg)                                # in_w = window hours; normal = normal-operation hours

on_dev = np.zeros(panel.M, bool); on_dev[dev] = True
print(f"window: H={cfg.H}, W={cfg.W} (lags {max(cfg.H, 1)}..{cfg.H + cfg.W - 1} h before each failure)")
print(f"dev machines:     {mk.in_w[on_dev].sum():>7,} window hours (positives) | {mk.normal[on_dev].sum():>7,} normal hours (negatives)")
print(f"holdout machines: {mk.in_w[~on_dev].sum():>7,} window hours             | {mk.normal[~on_dev].sum():>7,} normal hours")

window: H=0, W=48 (lags 1..47 h before each failure)
dev machines:      16,403 window hours (positives) | 393,313 normal hours (negatives)
holdout machines:  16,450 window hours             | 393,468 normal hours


## 2 · The features
32 features per machine-hour, all **causal**: each uses only the current and past hours, never the future. They
are the same building blocks the rules use (`src/detectors.py` §4):

| Group | Features | Count |
|---|---|---|
| Telemetry | for each of volt, rotate, pressure, vibration: the raw hourly reading, 6 h trailing mean, 24 h trailing mean, 24 h trailing standard deviation | 16 |
| Error log | for each of error1–error5: count in the last 24 h and in the last 48 h | 10 |
| Maintenance counters | hours since each component (comp1–comp4) was last maintained | 4 |
| Machine | age in years; model (1–4) | 2 |

In [4]:
F, names = feature_cube(panel, bank)        # shape: (machines, hours, features)
print("feature cube:", F.shape, "= machines x hours x features")

feature cube: (100, 8761, 32) = machines x hours x features


## 3 · Labels and training
* **Label:** an hour inside the 48 h window before a failure is **positive**. A normal-operation hour is
  **negative**. Other hours (recovery after a repair, warm-up, pre-failure hours of a different event's window)
  are left out, exactly as for the rules.
* **Training data:** dev machines only. Normal hours outnumber window hours ~50:1, so a random sample of
  negatives (40 × the positives) is used. The model only needs to *rank* hours, so class weights are left at the
  default.
* **Three models:**
  - **Gradient boosting** (`HistGradientBoostingClassifier`, 200 rounds, learning rate 0.08), all 32 features;
  - the **same without the 4 maintenance counters**. The first run leaned on them heavily, and notebook 05 shows
    why that is circular here;
  - **logistic regression** (standardised features), as a simple linear baseline.
* **Output:** each model gives every machine-hour a score between 0 and 1. The first 24 h (feature warm-up) are
  blank.

The negative sample is random, so F1 can shift by about ±0.02 between runs.

In [5]:
rng = np.random.default_rng(cfg.seed)
MODELS = {"hgb": "gradient boosting, all features",
          "hgb_nocounter": "gradient boosting, no maintenance counters",
          "logreg": "logistic regression"}
scores, importances = {}, {}
for kind, label in MODELS.items():
    t0 = time.time()
    # trains on dev machines, scores every machine-hour, measures permutation importance on holdout machines
    scores[kind], _, importances[kind] = train_and_score(panel, bank, mk, dev, hold, rng, kind=kind)
    print(f"{label:45s} trained and scored in {time.time() - t0:.0f} s")

gradient boosting, all features               trained and scored in 51 s


gradient boosting, no maintenance counters    trained and scored in 37 s


logistic regression                           trained and scored in 4 s


## 4 · From scores to alerts, scored like every rule
A score is turned into yes/no alerts **exactly the way every rule's threshold is set**. An hour alerts if the
score is above the **99th percentile of the model's scores during normal operation** on dev machines. Then the
brief's metrics are computed on the **holdout machines**:
* lift, precision (per alert: an off → on switch) and coverage (per failure), and F1 of precision and coverage;
* alerts per machine per month, and how early the first alert comes;
* 95% intervals from resampling whole holdout machines.


In [6]:
models_of = panel.machines.model.to_numpy()
rows, fires = [], {}
for kind, label in MODELS.items():
    s = scores[kind]
    thr = thresholds(s, mk.normal, "global", 99.0, dev, models_of)     # P99 of normal-operation scores (dev)
    with np.errstate(invalid="ignore"):
        fire = (s > thr) & np.isfinite(s)                              # alert ON where the score is above it
    fires[kind] = fire
    st = machine_stats(fire, mk)                                       # per-machine counts for the metrics
    h, ci = summarize(st[hold]), bootstrap(st, hold, 1000, rng)        # holdout metrics + 95% intervals
    lt = lead_times(fire, mk, hold)
    rows.append(dict(detector=f"ML: {label}", F1=h["f1"], F1_CI=f"[{ci['f1'][0]:.2f}, {ci['f1'][1]:.2f}]",
                     precision=h["precision"], coverage=h["coverage"], lift=h["lift"],
                     alerts_per_machine_month=h["alerts_per_machine_month"],
                     median_lead_h=np.median(lt) if lt else None))
bench = pd.DataFrame(rows).sort_values("F1", ascending=False)
base_precision = summarize(machine_stats(fires["hgb"], mk)[hold])["base_precision"]
print(f"random-alarm precision on holdout: {base_precision:.1%}")
bench.round(3)

random-alarm precision on holdout: 4.1%


,detector,F1,F1_CI,precision,coverage,lift,alerts_per_machine_month,median_lead_h
2,ML: logistic regression,0.627,"[0.59, 0.66]",0.45739910313901344,0.997,56.791,2.757,24.0
1,"ML: gradient boosting, no maintenance counters",0.518,"[0.49, 0.55]",0.3491879350348028,1.000,42.562,4.567,43.0
0,"ML: gradient boosting, all features",0.306,"[0.27, 0.34]",0.1806420699568759,1.000,46.238,3.686,47.0


**Conclusion.**
* **Every model catches essentially every failure** (coverage ~100%), **but raises far more false alarms**:
  only 18–46% of their alerts are followed by a failure.
* **Logistic regression is the best model** (F1 ~0.63). It adds up evidence (recent errors plus drifting signals)
  without requiring that the drift and the error belong to the *same part*. Notebook 04's best rule (C09)
  requires exactly that pairing and reaches F1 0.92 (side by side in notebook 06, section 8).
* **Gradient boosting with all features is the worst** (F1 ~0.30). Section 5 shows it relies almost entirely on
  the maintenance counters, which look predictive in training but don't hold up on holdout machines.

## 5 · What the models lean on
**Permutation importance:** shuffle one feature at a time and measure how much the model's ranking of window vs
normal hours (average precision) drops. Measured on **holdout** machines, so it reflects what generalises rather
than what the model memorised.

In [7]:
# Top 10 features per model (bigger = the model relies on it more).
top = {MODELS[k]: pd.Series(v).sort_values(ascending=False).head(10) for k, v in importances.items()}
for label, s in top.items():
    print(f"\n{label}:")
    print(s.round(3).to_string())


gradient boosting, all features:
hours_since_maint_comp2    0.178
hours_since_maint_comp3    0.143
hours_since_maint_comp1    0.141
hours_since_maint_comp4    0.138
pressure_roll6             0.035
vibration_roll6            0.024
rotate_roll6               0.018
volt_roll24                0.018
machine_age_years          0.016
machine_model              0.014

gradient boosting, no maintenance counters:
pressure_roll6       0.098
rotate_roll6         0.071
volt_roll24          0.070
error3_count24       0.069
machine_model        0.056
vibration_roll6      0.055
error5_count24       0.054
error1_count24       0.046
machine_age_years    0.040
volt_roll6           0.040

logistic regression:
error5_count24     0.087
error4_count24     0.068
error1_count24     0.053
error3_count24     0.044
rotate_roll6       0.043
volt_roll24        0.040
vibration_roll6    0.039
pressure_roll6     0.032
error2_count24     0.030
rotate_roll24      0.022


**Conclusion.**
* **With counters, gradient boosting leans almost entirely on "hours since maintenance"** (its top 4 features).
  In this labelling "normal operation" is defined using the failures (it excludes the hours before them), so a
  counter's normal range quietly encodes where failures are. The model learns that circularity, and it does not
  carry over to the holdout machines. This is the same trap the causal gate catches in the counter rules
  (notebook 05, section 3).
* **Without counters, both models rely on the same ingredients as our rules:**
  - smoothed signals: the **6 h and 24 h trailing means** of pressure, rotation, voltage and vibration;
  - **24 h error counts**, especially error5, error4, error1 and error3 (each part's own codes).

  That confirmed the brainstorm's direction (notebook 02) toward smoothed drifts combined with error codes.
* **The models found no new signal family.** The raw hourly readings, the 24 h spread and the machine's age or
  model matter little. That is why the candidate list did not grow beyond those families.

## 6 · Save this notebook's results
The model scores and the features each model leans on are saved to `outputs/03_ml_benchmark.json` for
notebook 06.

In [8]:
save_results("03_ml_benchmark", dict(                         # src/evaluation.py §9
    models=bench,                                              # holdout scores of each model (section 4)
    random_alarm_precision=base_precision,
    top_features={label: s.round(4) for label, s in top.items()},   # permutation importance (section 5)
), cfg)

saved outputs/03_ml_benchmark.json


PosixPath('/Users/avijay/Documents/oden_precursors/outputs/03_ml_benchmark.json')